# Hands-on P02: Teori Neural Network

**Durasi Hands-on:** ±58 menit (6 langkah selaras Slide 36–41 + Eksperimen Opsional)  
**Dataset Proyek Utama:** **Disney Frozen Figurine (`data/frozen_v1/`)** — 5 kelas (`anna, elsa, kristoff, olaf, sven`), dibaca secara **dinamis** dari `DATA_DIR` dan `metadata.csv`.  
**Gaya Kode:** Ramah pemula — setiap sel pendek (5–15 baris), 1 sel = 1 tugas, lengkap dengan komentar Bahasa Indonesia (`# ...`).

**Peta 6 Langkah Praktik Hari Ini (100% Selaras Bab 1–3):**
1. **Langkah 1 (Slide 03 & 36):** Setup 3 tahap & membaca data Frozen secara dinamis dari `metadata.csv`.
2. **Langkah 2 (Slide 21–22 & 37):** *Flatten* foto Frozen & menghitung neuron **Input Layer** (`H × W × C`) serta parameter 1-layer (`752.645`).
3. **Langkah 3 (Slide 25–29 & 38):** *Forward Pass* manual di **NumPy** (1 neuron *Weight & Bias*, `ReLU`, `Softmax`, `Argmax`, & *Cross-Entropy Loss*).
4. **Langkah 4 (Slide 29 & 39):** Membangun **Neural Network 1-Layer** (`Input → Flatten → Dense(5, softmax)`) dengan `tf.keras.Sequential`.
5. **Langkah 5 (Slide 28–29 & 40):** Melatih model (`model.fit()`) & memplot kurva penurunan **Loss** per *Epoch*.
6. **Langkah 6 (Slide 29 & 41):** Mencocokkan `model.summary()` dengan hitungan tangan `(Input × Output) + Bias`, plus eksperimen opsional **Learning Rate**.

> **Cara Menjalankan:** Tekan `Shift + Enter` berurutan dari atas ke bawah. Pada versi peserta, lengkapi bagian bertanda `# TODO`.

## Setup 1/3: Memanggil Pustaka (*Libraries*) Python
Kita panggil alat-alat Python yang diperlukan dan mencetak versinya agar eksperimen dapat direproduksi (*reproducible*).

In [ ]:
import os                           # Untuk membaca pengaturan komputer / environment
import random                       # Untuk mengunci angka acak bawaan Python
from pathlib import Path            # Untuk mengatur alamat folder & file

import numpy as np                  # NumPy: untuk mengolah matriks/tabel angka
import pandas as pd                 # Pandas: untuk membuat tabel laporan
import matplotlib.pyplot as plt     # Matplotlib: untuk menampilkan gambar & grafik
import tensorflow as tf             # TensorFlow/Keras: untuk membangun & melatih Neural Network
from PIL import Image               # Pillow: untuk membuka dan me-resize foto Frozen

print("✅ Semua library berhasil dimuat!")
print("Versi NumPy      :", np.__version__)
print("Versi TensorFlow :", tf.__version__)

## Setup 2/3: Deteksi Google Colab & Mengunci `SEED = 42`
Notebook ini bisa berjalan di **Google Colab** maupun di **komputer lokal**. Kita kunci `SEED = 42` agar hasil acak selalu konsisten setiap kali dijalankan ulang.

In [ ]:
# Cek apakah notebook dijalankan di Google Colab atau lokal
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Kunci angka acak (SEED = 42) untuk reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Berjalan di Google Colab :", IN_COLAB)
print("SEED dikunci pada        :", SEED)

## Setup 3/3: Konfigurasi Proyek & Alamat Folder
Daftar kelas, ukuran gambar (`IMG_SIZE = 224`), dan folder data disimpan sebagai variabel konfigurasi di sini (tidak di-*hardcode*) agar nanti mudah diganti ke foto Disney buatan sendiri.

In [ ]:
# Konfigurasi utama proyek (dinamis — tetap aman jika jumlah foto dataset ditambah nanti)
IMG_SIZE = 224
SMOKE_TEST = os.environ.get("SMOKE_TEST", "0") == "1"
CLASS_NAMES = ["anna", "elsa", "kristoff", "olaf", "sven"]


# Cari folder utama proyek yang memiliki folder "data/"
BASE_DIR = Path(".")
for folder in [Path("."), Path(".."), Path("../.."), Path("../../..")]:
    if (folder / "data").exists():
        BASE_DIR = folder
        break

# Gunakan dataset proyek utama Disney Frozen (otomatis fallback ke data/sample jika belum tersedia)
DATA_DIR = BASE_DIR / "data" / "frozen_v1"
if not DATA_DIR.exists():
    DATA_DIR = BASE_DIR / "data" / "sample"
RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Mode SMOKE_TEST  :", SMOKE_TEST)
print("Folder Dataset   :", DATA_DIR.resolve())
print("Folder Hasil     :", RESULTS_DIR.resolve())

---
## Langkah 1 (9,5 menit): Membaca Dataset Frozen Dinamis (Sesuai Slide 03 & 36)

Kita tidak pernah meng-hardcode jumlah foto karena dataset bisa terus bertambah di versi berikutnya.  
Mari baca tabel `metadata.csv` langsung dari `DATA_DIR` dan cek distribusi tiap karakter (`polos` vs `ramai`).

In [ ]:
# Baca metadata dataset Frozen secara dinamis
meta_path = DATA_DIR / "metadata.csv"
df_meta = pd.read_csv(meta_path)

print("Total foto tersedia saat ini :", len(df_meta))
print("Daftar kelas di metadata     :", sorted(df_meta["kelas"].unique().tolist()))
print("\nTabel jumlah foto per Kelas x Kondisi:")
print(pd.crosstab(df_meta["kelas"], df_meta["kondisi"], margins=True))

Mari tampilkan 1 contoh foto nyata untuk masing-masing dari 5 karakter Frozen (`anna`, `elsa`, `kristoff`, `olaf`, `sven`):

In [ ]:
# Tampilkan 1 foto polos untuk tiap karakter Frozen
fig, axes = plt.subplots(1, len(CLASS_NAMES), figsize=(12, 2.8))

for i, kls in enumerate(CLASS_NAMES):
    baris = df_meta[(df_meta["kelas"] == kls) & (df_meta["kondisi"] == "polos")].iloc[0]
    foto = Image.open(DATA_DIR / baris["file"]).convert("RGB")
    axes[i].imshow(foto)
    axes[i].set_title(kls.capitalize(), fontweight="bold")
    axes[i].axis("off")

plt.suptitle("5 Karakter Proyek Akhir: Image Classification (Multi-Kelas)", fontweight="bold")
plt.tight_layout()
plt.show()

---
## Langkah 2 (9,5 menit): *Flatten* Foto & Hitung Neuron *Input Layer* (Sesuai Slide 21–22 & 37)

Ingat aturan penting di **Slide 22–23**:
- Jumlah **neuron** di *Input Layer* sama dengan banyaknya nilai piksel dari **1 gambar** setelah *resize*: `Tinggi × Lebar × Channel`.
- Agar bisa masuk ke *Input Layer*, gambar 3D `(H, W, 3)` diluruskan menjadi 1 vektor panjang lewat operasi **Flatten**.

In [ ]:
# Ambil 1 foto asli Elsa, resize ke 64x64 dan 224x224, lalu normalisasi ke 0.0-1.0
baris_elsa = df_meta[(df_meta["kelas"] == "elsa") & (df_meta["kondisi"] == "polos")].iloc[0]
foto_elsa = Image.open(DATA_DIR / baris_elsa["file"]).convert("RGB")

arr_64 = np.array(foto_elsa.resize((64, 64)), dtype=np.float32) / 255.0
arr_224 = np.array(foto_elsa.resize((IMG_SIZE, IMG_SIZE)), dtype=np.float32) / 255.0

print("Shape foto 64x64   :", arr_64.shape, "| min/max:", round(float(arr_64.min()), 2), "/", round(float(arr_64.max()), 2))
print("Shape foto 224x224 :", arr_224.shape)

### Latihan 1 (`# TODO 1`): *Flatten* Foto `64×64×3` Menjadi Vektor Input Layer
Gunakan `.flatten()` pada `arr_64` untuk mengubah array 3D `(64, 64, 3)` menjadi 1 deretan neuron input, lalu cek panjangnya (`len(vektor_64)`).

In [ ]:
# TODO 1: Luruskan arr_64 menjadi vektor 1D dengan .flatten()
vektor_64 = None  # TODO: arr_64.flatten()

assert vektor_64 is not None and vektor_64.ndim == 1
assert len(vektor_64) == 64 * 64 * 3
print("Shape setelah flatten :", vektor_64.shape)
print("Jumlah neuron input   :", f"{len(vektor_64):,}")
print("✅ Cek Latihan 1 lolos!")

<details>
<summary><b>💡 Hint &amp; Jawaban Latihan 1</b> (klik untuk membuka)</summary>

```python
vektor_64 = arr_64.flatten()
```
</details>

### Latihan 2 (`# TODO 2`): Hitung Neuron Input `224×224×3` & Total Parameter `W + b`
Hitung `neuron_input_224 = IMG_SIZE * IMG_SIZE * 3` dan `param_1layer_224 = neuron_input_224 * len(CLASS_NAMES) + len(CLASS_NAMES)` — cocokkan dengan angka **`752.645`** di Slide 29!

In [ ]:
# TODO 2A: Hitung jumlah neuron input untuk foto 224x224x3
neuron_input_224 = None  # TODO: IMG_SIZE * IMG_SIZE * 3

# TODO 2B: Hitung total parameter (weight + bias) menuju 5 neuron output
param_1layer_224 = None  # TODO: neuron_input_224 * len(CLASS_NAMES) + len(CLASS_NAMES)

assert neuron_input_224 == 150_528
assert param_1layer_224 == 752_645
print("Neuron Input (224x224x3) :", f"{neuron_input_224:,}")
print("Total Param (W + b)      :", f"{param_1layer_224:,}")
print("✅ Cek Latihan 2 lolos!")

<details>
<summary><b>💡 Hint &amp; Jawaban Latihan 2</b> (klik untuk membuka)</summary>

```python
neuron_input_224 = IMG_SIZE * IMG_SIZE * 3
param_1layer_224 = neuron_input_224 * len(CLASS_NAMES) + len(CLASS_NAMES)
```
</details>

---
## Langkah 3 (10 menit): *Forward Pass* Manual dengan NumPy (Sesuai Slide 25–29 & 38)

Sekarang mari buktikan sendiri bahwa matematika di dalam *Neural Network* hanyalah perkalian bobot + bias (`z = Σ(x·w) + b`), fungsi aktivasi (`ReLU` / `Softmax`), `Argmax`, dan `Loss (-ln P)`!

### Latihan 3 (`# TODO 3`): Menghitung 1 Neuron (`W·x + b`) & Aktivasi `ReLU`
Lengkapi perhitungan `z = float(np.dot(x_rgb, w) + b)` dan fungsi `relu(v)` yang mengembalikan `np.maximum(0.0, v)` (sesuai Slide 25–26).

In [ ]:
# Contoh input 3 angka [R, G, B] dan bobot 1 neuron
x_rgb = np.array([0.67, 0.66, 0.67], dtype=np.float32)
w = np.array([-1.5, 1.5, 2.5], dtype=np.float32)
b = -1.3

# TODO 3A: Hitung skor z = float(np.dot(x_rgb, w) + b)
z = None  # TODO

def relu(v):
    # TODO 3B: Kembalikan np.maximum(0.0, v)
    return None  # TODO

out_relu = relu(z) if z is not None else None
assert z is not None and np.isclose(z, 0.36, atol=1e-2)
assert out_relu is not None and np.isclose(out_relu, z, atol=1e-6)
assert relu(-2.5) == 0.0
print(f"Skor z = {z:.4f} -> ReLU(z) = {out_relu:.4f}")
print("✅ Cek Latihan 3 lolos!")

<details>
<summary><b>💡 Hint &amp; Jawaban Latihan 3</b> (klik untuk membuka)</summary>

```python
z = float(np.dot(x_rgb, w) + b)
# dan di dalam relu(v):
return np.maximum(0.0, v)
```
</details>

### Latihan 4 (`# TODO 4`): `softmax_manual`, `Argmax` & *Cross-Entropy Loss*
Lengkapi `softmax_manual(logits)` memakai `exp_val = np.exp(shifted)` lalu `exp_val / np.sum(exp_val)`. Setelah itu kita jalankan Forward Pass 1-Layer lengkap (`W·x + b → Softmax → Argmax → Loss`) pada 4 piksel nyata Elsa!

In [ ]:
def softmax_manual(logits):
    shifted = logits - np.max(logits)
    # TODO 4: Hitung exp_val dan bagi dengan jumlah totalnya
    exp_val = None  # TODO: np.exp(shifted)
    return None     # TODO: exp_val / np.sum(exp_val)

# Forward Pass 1-Layer nyata pada 4 piksel Elsa -> 5 kelas Frozen (persis alur Slide 25-29!)
x_4px = np.array([0.76, 0.74, 0.65, 0.65], dtype=np.float64)
W_demo = np.array([
    [-0.4,  1.2, -0.3, -0.5, -0.2],
    [-0.3,  1.1, -0.2, -0.4, -0.3],
    [-0.2,  0.9, -0.4, -0.3, -0.1],
    [-0.3,  0.8, -0.2, -0.2, -0.3],
])
b_demo = np.array([-0.1, -0.5, 0.1, 0.0, -0.1])

z_skor = x_4px @ W_demo + b_demo
probs = softmax_manual(z_skor)

assert probs is not None and np.isclose(np.sum(probs), 1.0)
pred_idx = int(np.argmax(probs))
loss_elsa = float(-np.log(probs[1]))  # Target Elsa = index 1
print("Skor mentah z    :", np.round(z_skor, 2))
print("Probabilitas (%) :", np.round(probs * 100, 1))
print(f"Argmax Prediksi  : {CLASS_NAMES[pred_idx].upper()} ({probs[pred_idx]*100:.1f}%) | Loss = {loss_elsa:.2f}")
print("✅ Cek Latihan 4 lolos!")

<details>
<summary><b>💡 Hint &amp; Jawaban Latihan 4</b> (klik untuk membuka)</summary>

```python
exp_val = np.exp(shifted)
return exp_val / np.sum(exp_val)
```
</details>

---
## Langkah 4 (10 menit): Membangun Neural Network Pertamamu di Keras (Sesuai Slide 29 & 39)

Mari muat seluruh foto Frozen dari `DATA_DIR` ke ukuran ringkas `64×64×3` (agar RAM tetap ringan di kelas), lalu bangun **Neural Network 1-Layer** persis seperti diagram Slide 29: `Input → Flatten → Dense(5, activation="softmax")`!

> **Catatan Jujur:** Model 1-layer ini adalah langkah awal agar kamu paham penuh alur piksel → bobot → Softmax → Loss. Di **P03** kita akan naik ke arsitektur berlapis & **CNN** yang jauh lebih kuat mengenali pola 2D foto!

In [ ]:
# Muat foto Frozen ke ukuran 64x64 RGB (skala 0.0-1.0) secara dinamis
SMALL_SIZE = 64
label_to_idx = {nama: i for i, nama in enumerate(CLASS_NAMES)}

X_frozen, y_frozen = [], []
for _, row in df_meta.iterrows():
    im = Image.open(DATA_DIR / row["file"]).convert("RGB").resize((SMALL_SIZE, SMALL_SIZE))
    X_frozen.append(np.array(im, dtype=np.float32) / 255.0)
    y_frozen.append(label_to_idx[row["kelas"]])

X_frozen = np.array(X_frozen)
y_frozen = np.array(y_frozen)
print("Shape X_frozen :", X_frozen.shape, "| Jumlah label:", len(y_frozen))

### Latihan 5 (`# TODO 5`): Menyusun Neural Network 1-Layer (`buat_nn_frozen`)
Lengkapi fungsi `buat_nn_frozen(lr=0.001)` dengan urutan persis Slide 29: `Input((SMALL_SIZE, SMALL_SIZE, 3))` → `Flatten()` → `Dense(len(CLASS_NAMES), activation='softmax')`.

In [ ]:
def buat_nn_frozen(lr=0.001):
    # TODO 5: Lengkapi layer Flatten dan Output Dense(5, softmax)
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(SMALL_SIZE, SMALL_SIZE, 3)),
        None,  # TODO: tf.keras.layers.Flatten()
        None,  # TODO: tf.keras.layers.Dense(len(CLASS_NAMES), activation="softmax")
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

model_nn = buat_nn_frozen(lr=0.001)
assert model_nn.output_shape == (None, len(CLASS_NAMES))
print("✅ Neural Network 1-Layer berhasil dibangun! Output shape:", model_nn.output_shape)

<details>
<summary><b>💡 Hint &amp; Jawaban Latihan 5</b> (klik untuk membuka)</summary>

```python
tf.keras.layers.Flatten(),
tf.keras.layers.Dense(len(CLASS_NAMES), activation="softmax"),
```
</details>

---
## Langkah 5 (10 menit): Latih Model (`model.fit()`) & Plot Kurva *Loss* (Sesuai Slide 28–29 & 40)

Sekarang kita panggil `model_nn.fit(...)` untuk melihat langsung siklus **Forward Pass → Loss → Backpropagation → Gradient Descent** bekerja otomatis di Keras!  
*(Catatan metodologi: pembagian resmi `train/val/test` bebas data leakage dengan Group Split akan kita pelajari tuntas di **P05**.)*

### Latihan 6 (`# TODO 6`): Menjalankan `model_nn.fit(...)`
Latih `model_nn` memakai `X_frozen` dan `y_frozen` selama `EPOCHS` (`2` saat `SMOKE_TEST`, `8` saat kelas normal) dengan `batch_size=16`.

In [ ]:
EPOCHS = 2 if SMOKE_TEST else 8

# TODO 6: Panggil model_nn.fit(X_frozen, y_frozen, epochs=EPOCHS, batch_size=16, verbose=1)
history = None  # TODO

assert history is not None and "loss" in history.history
print(f"Loss Epoch 1     : {history.history['loss'][0]:.4f}")
print(f"Loss Epoch akhir : {history.history['loss'][-1]:.4f}")
print("✅ Cek Latihan 6 lolos!")

<details>
<summary><b>💡 Hint &amp; Jawaban Latihan 6</b> (klik untuk membuka)</summary>

```python
history = model_nn.fit(X_frozen, y_frozen, epochs=EPOCHS, batch_size=16, verbose=1)
```
</details>

Mari gambar grafik *Training Loss* per *Epoch* dan simpan ke folder `results/`:

In [ ]:
# Plot kurva Training Loss Neural Network 1-Layer pada foto Frozen (demo konsep)
fig, ax = plt.subplots(figsize=(7.0, 3.6))
epoch_x = range(1, len(history.history["loss"]) + 1)
ax.plot(epoch_x, history.history["loss"], marker="o", color="#4285F4", linewidth=2.5, label="Training Loss")
ax.set_title("Kurva Training Loss — Neural Network 1-Layer pada Foto Frozen (64×64×3)")
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-Entropy Loss (makin kecil makin baik)")
ax.set_xticks(list(epoch_x))
ax.grid(True, linestyle="--", alpha=0.4)
ax.legend()
plt.tight_layout()

loss_plot_path = RESULTS_DIR / "p02_nn_loss_curve.png"
fig.savefig(loss_plot_path, dpi=150)
plt.show()
assert loss_plot_path.exists()
print("Grafik disimpan ke:", loss_plot_path)

---
## Langkah 6 (9 menit): Cocokkan `model.summary()` dengan Hitungan Tangan (Sesuai Slide 29 & 41)

Mari cetak tabel `model_nn.summary()` lalu buktikan bahwa angka di kolom `Param #` cocok **100%** dengan rumus `(Input × Output) + Bias` dari Bab 3!

In [ ]:
# Tampilkan ringkasan arsitektur & jumlah parameter dari Keras
model_nn.summary()

### Latihan 7 (`# TODO 7`): Membuktikan Hitungan Tangan == `model_nn.count_params()`
Hitung `n_weight = n_input * n_kelas` dan `n_bias = n_kelas`, lalu jumlahkan ke `total_manual` dan pastikan sama persis dengan `model_nn.count_params()` (`61.445`)!

In [ ]:
n_input = SMALL_SIZE * SMALL_SIZE * 3   # 12.288 neuron input
n_kelas = len(CLASS_NAMES)              # 5 neuron output

# TODO 7A: Hitung jumlah Weight (Input * Output)
n_weight = None  # TODO: n_input * n_kelas
# TODO 7B: Hitung jumlah Bias (= jumlah neuron output)
n_bias = None    # TODO: n_kelas

total_manual = (n_weight + n_bias) if (n_weight is not None and n_bias is not None) else None
total_keras = model_nn.count_params()

assert total_manual == total_keras, f"Belum cocok: manual={total_manual}, keras={total_keras}"
print(f"Jumlah Weight (12.288 x 5) : {n_weight:>10,}")
print(f"Jumlah Bias (5 kelas)      : {n_bias:>10,}")
print(f"Total Hitungan Tangan      : {total_manual:>10,}")
print(f"Total Keras count_params() : {total_keras:>10,}  <-- COCOK 100%!")
print("✅ Cek Latihan 7 lolos!")

<details>
<summary><b>💡 Hint &amp; Jawaban Latihan 7</b> (klik untuk membuka)</summary>

```python
n_weight = n_input * n_kelas
n_bias = n_kelas
```
</details>

---
## Eksperimen Tambahan (Opsional): Pengaruh *Learning Rate* (`0.0001` vs `0.001` vs `0.01`)

Ingat **Tahap 5 (Backpropagation & Gradient Descent)** di Bab 3: **Learning Rate** mengatur seberapa besar langkah koreksi bobot `W & b` tiap langkah!  
Lengkapi `# TODO 8` untuk membandingkan 3 nilai *Learning Rate* berbeda pada dataset Frozen kita!

### Latihan 8 (`# TODO 8` · Opsional): Eksperimen 3 Nilai *Learning Rate*
Di dalam loop `daftar_lr = [0.0001, 0.001, 0.01]`, panggil `buat_nn_frozen(lr=lr_val)` lalu latih selama `EPOCHS` dan catat *Loss* awal vs *Loss* akhirnya.

In [ ]:
daftar_lr = [0.0001, 0.001, 0.01]
hasil_eksperimen = []

for lr_val in daftar_lr:
    # TODO 8: Bangun model memakai buat_nn_frozen(lr=lr_val)
    m_uji = None  # TODO
    assert m_uji is not None
    h_uji = m_uji.fit(X_frozen, y_frozen, epochs=EPOCHS, batch_size=16, verbose=0)
    hasil_eksperimen.append({
        "learning_rate": lr_val,
        "loss_awal": round(float(h_uji.history["loss"][0]), 4),
        "loss_akhir": round(float(h_uji.history["loss"][-1]), 4),
        "total_parameter": int(m_uji.count_params()),
    })

df_exp = pd.DataFrame(hasil_eksperimen)
exp_csv = RESULTS_DIR / "p02_lr_experiment.csv"
df_exp.to_csv(exp_csv, index=False)

assert len(df_exp) == 3 and exp_csv.exists()
param_224 = (IMG_SIZE * IMG_SIZE * 3 * len(CLASS_NAMES)) + len(CLASS_NAMES)
print(df_exp.to_string(index=False))
print(f"\nSebagai perbandingan — 1-Layer 224x224x3: {param_224:,} parameter (persis Slide 29!)")
print("✅ Seluruh Hands-on P02 selesai!")

<details>
<summary><b>💡 Hint &amp; Jawaban Latihan 8</b> (klik untuk membuka)</summary>

```python
m_uji = buat_nn_frozen(lr=lr_val)
```
</details>

## Referensi Belajar Mandiri (Selaras Slide 33–34)

1. [3Blue1Brown — *But what is a Neural Network?*](https://www.youtube.com/watch?v=aircAruvnKk) *(Wajib)*
2. [3Blue1Brown — *Playlist Neural Networks (Bab 1–4)*](https://www.youtube.com/playlist?list=PLZHQObOWTQDNU6R1_67000Dx_ZCJB-3pi)
3. [3Blue1Brown — *Versi Tulis Neural Networks*](https://3blue1brown.com/lessons/neural-networks)
4. [TensorFlow Playground](https://playground.tensorflow.org/) `[CEK LINK]`
5. [Google Machine Learning Crash Course](https://developers.google.com/machine-learning/crash-course) `[CEK LINK]`
6. [Keras Official — *The Sequential Model Guide*](https://keras.io/guides/sequential_model/) `[CEK LINK]`
7. [Coursera (DeepLearning.AI) — *Convolutional Neural Networks (Preview P03)*](https://www.coursera.org/learn/convolutional-neural-networks)
8. [CNN Explainer (Visual Interaktif)](https://poloclub.github.io/cnn-explainer/) `[CEK LINK]`
9. [Stanford CS231n — *Lecture Notes*](https://cs231n.github.io/) `[CEK LINK]`
10. [He dkk. — *Deep Residual Learning (ResNet)*](https://arxiv.org/abs/1512.03385) `[CEK LINK]`
11. [Sandler dkk. — *MobileNetV2: Inverted Residuals and Linear Bottlenecks*](https://arxiv.org/abs/1801.04381)
12. [TensorFlow — *Transfer Learning & Fine-Tuning (Versi Indonesia)*](https://www.tensorflow.org/tutorials/images/transfer_learning?hl=id)